In [ ]:
import sys
import plotly.graph_objects as go
from pathlib import Path
import pandas as pd
import numpy as np
import re
# --- Data Loading ---

files = [
    'prices_round_4_day_1.csv',
    'prices_round_4_day_2.csv',
    'prices_round_4_day_3.csv'
]

prices = []
for file in files:
    try:
        df = pd.read_csv(file, sep=';')
        day_match = re.search(r'day_(-?\d+)', file)
        if day_match:
            df['day'] = int(day_match.group(1))
        prices.append(df)
    except FileNotFoundError:
        print(f"Warning: {file} not found.")

df_total = pd.concat(prices, ignore_index=True)
df_total = df_total.sort_values(by=['day', 'timestamp']).reset_index(drop=True)

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Parameters
ALPHA = 0.01
WINDOW = 100
Z_THRESHOLD = 2.3
products = ["HYDROGEL_PACK"]

days = [0, 1, 2]

# Assuming df_total exists in your environment
for day in days:
    subset = df_total[(df_total['product'].isin(products)) & (df_total['day'] == day)].copy()
    
    if subset.empty: 
        continue

    # 1. Clean data and handle zeros
    subset['mid_price'] = subset['mid_price'].replace(0, np.nan).ffill()
    
    for product in products:
        prod_data = subset[subset['product'] == product].copy().sort_values('timestamp')
        
        # --- Optimized Signal Calculation ---
        # Use Pandas built-ins for EWMA and Rolling Std (much more robust)
        prod_data['ewma_mean'] = prod_data['mid_price'].ewm(alpha=ALPHA, adjust=False).mean()
        
        # Calculate rolling volatility of the price relative to the mean
        # We use the rolling standard deviation of the price to normalize the Z-Score
        prod_data['rolling_std'] = prod_data['mid_price'].rolling(window=WINDOW).std()
        
        # Z-Score: (Current - Mean) / Volatility
        prod_data['z_score'] = (prod_data['mid_price'] - prod_data['ewma_mean']) / prod_data['rolling_std']
        
        # Fill NaNs from the initial window so Plotly doesn't skip the first 50 points
        prod_data['z_score'] = prod_data['z_score'].fillna(0)

        # --- Enhanced Graphing ---
        fig = make_subplots(
            rows=2, cols=1, 
            shared_xaxes=True, 
            vertical_spacing=0.08, 
            row_heights=[0.7, 0.3],
            subplot_titles=(f"{product} Price Heatmap", "Z-Score Signal")
        )

        # Row 1: The "Heatmap" Overlay
        # Trace 1: The background line (thin and light)
        fig.add_trace(go.Scatter(
            x=prod_data['timestamp'], y=prod_data['mid_price'],
            mode='lines', name='Price Path',
            line=dict(color='rgba(150, 150, 150, 0.3)', width=1)
        ), row=1, col=1)

        fig.add_trace(go.Scatter(
            x=prod_data['timestamp'], y=prod_data['ewma_mean'],
            mode='lines', name='Price Path',
        ), row=1, col=1)

        # Trace 2: The colored markers (The actual signal)
        fig.add_trace(go.Scatter(
            x=prod_data['timestamp'], y=prod_data['mid_price'],
            mode='markers',
            name='Z-Score Color',
            marker=dict(
                size=6,
                color=prod_data['z_score'],
                colorscale='RdBu', 
                reversescale=True,  # Red = High/Overbought, Blue = Low/Oversold
                cmid=0,             # Ensure 0 is the neutral center color
                cmin=-Z_THRESHOLD,  # Cap color intensity at your threshold
                cmax=Z_THRESHOLD,
                showscale=True,
                colorbar=dict(title="Z-Score", x=1.02, len=0.7)
            ),
            hovertext=[f"Price: {p:.2f}<br>Z: {z:.2f}" for p, z in zip(prod_data['mid_price'], prod_data['z_score'])],
            hoverinfo="text+x"
        ), row=1, col=1)

        # Row 2: Standard Z-Score Line
        fig.add_trace(go.Scatter(
            x=prod_data['timestamp'], y=prod_data['z_score'],
            name='Z-Score Value', 
            line=dict(color='black', width=1.2)
        ), row=2, col=1)

        # Threshold lines
        for val in [Z_THRESHOLD, -Z_THRESHOLD]:
            fig.add_hline(y=val, line_dash="dash", line_color="red", row=2, col=1)

        fig.update_layout(
            height=800, 
            template='plotly_white', 
            title_text=f"Market Analysis: {product} (Day {day})",
            showlegend=False
        )
        
        fig.show()

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Configuration ---
# Kalman Gain Control: 
# Increase R to make the Local Mean "stiffer" (slower to move)
# Increase Q to make the Local Mean "faster" (track spikes more closely)
Q_PROCESS = 1e-5 
R_MEASURE = 0.05 

WINDOW = 50      # For volatility calculation
Z_THRESHOLD = 2.0
products = ["HYDROGEL_PACK"]

def apply_kalman_local_mean(series, q=1e-5, r=0.01, global_mean=10000):
    """Tracks the 'Local' consensus price, ignoring micro-noise."""
    means = []
    curr_est = series.iloc[0] if not series.empty else global_mean
    error_est = 1.0
    for val in series:
        # Prediction step
        error_est += q
        # Update step
        kalman_gain = error_est / (error_est + r)
        curr_est = curr_est + kalman_gain * (val - curr_est)
        error_est = (1 - kalman_gain) * error_est
        means.append(curr_est)
    return pd.Series(means, index=series.index)

for day in [1, 2, 3]:
    subset = df_total[(df_total['product'].isin(products)) & (df_total['day'] == day)].copy()
    if subset.empty: continue
    
    # Cleaning
    subset['mid_price'] = subset['mid_price'].replace(0, np.nan).ffill()
    
    for product in products:
        prod_data = subset[subset['product'] == product].copy().sort_values('timestamp')

        global_mean = prod_data['mid_price'].mean() # Could also be a fixed value like 10k
        
        # 1. Local Anchor (The Kalman Filter)
        # This captures the "smaller mean reversions" around spikes
        prod_data['local_mean'] = apply_kalman_local_mean(prod_data['mid_price'], Q_PROCESS, R_MEASURE, global_mean)
        
        # 2. Volatility (Standard Deviation around the Local Mean)
        # We use a floor (1e-2) to prevent the Z-score from exploding in flat markets
        prod_data['rolling_std'] = prod_data['mid_price'].rolling(window=WINDOW).std().replace(0, np.nan).ffill().fillna(1.0)
        
        # 3. The Signal: Deviation from Local Mean
        prod_data['z_score'] = (prod_data['mid_price'] - global_mean) / prod_data['rolling_std']
        
        # 4. Global Bias: How far is our "Local Consensus" from the 10k "True North"?
        prod_data['global_bias'] = prod_data['local_mean'] - global_mean

        # --- Visualization ---
        fig = make_subplots(
            rows=2, cols=1, shared_xaxes=True, 
            vertical_spacing=0.05, row_heights=[0.7, 0.3],
            subplot_titles=(f"{product} Local vs Global Mean", "Reversion Signal (Z-Score)")
        )

        # Main Plot: Price & Means
        fig.add_trace(go.Scatter(x=prod_data['timestamp'], y=prod_data['mid_price'], 
                                 line=dict(color='rgba(200,200,200,0.4)', width=1), name='Market Price'), row=1, col=1)
        
        # Global  Anchor
        fig.add_hline(y=global_mean, line_dash="dot", line_color="black", annotation_text=f"Global Anchor {global_mean:.2f} ", row=1, col=1)
        
        # Local Kalman Mean
        fig.add_trace(go.Scatter(x=prod_data['timestamp'], y=prod_data['local_mean'], 
                                 line=dict(color='orange', width=2), name='Local Mean (Kalman)'), row=1, col=1)

        # Heatmap Markers
        fig.add_trace(go.Scatter(
            x=prod_data['timestamp'], y=prod_data['mid_price'],
            mode='markers', name='Trade Signal',
            marker=dict(
                size=5, color=prod_data['z_score'], colorscale='RdBu', reversescale=True,
                cmid=0, cmin=-Z_THRESHOLD, cmax=Z_THRESHOLD, showscale=True
            )
        ), row=1, col=1)

        # Z-Score Subplot
        fig.add_trace(go.Scatter(x=prod_data['timestamp'], y=prod_data['z_score'], 
                                 line=dict(color='black', width=1), name='Z-Score'), row=2, col=1)
        
        for v in [Z_THRESHOLD, -Z_THRESHOLD]:
            fig.add_hline(y=v, line_dash="dash", line_color="red", row=2, col=1)

        fig.update_layout(height=900, template='plotly_white', title_text=f"Hybrid Analysis: {product} (Day {day})", showlegend=False)
        fig.show()

In [ ]:
# Calculate mean price for each product across all 3 days
product_means = df_total.groupby('product')['mid_price'].mean().sort_values(ascending=False)
print("Mean Price by Product (All Days Combined):")
print(product_means)
print(f"\nTotal products: {len(product_means)}")

Mean Price by Product (All Days Combined):
product
HYDROGEL_PACK          9990.806867
VELVETFRUIT_EXTRACT    5250.098100
VEV_4000               1250.109800
VEV_4500                750.109567
VEV_5000                255.022400
VEV_5100                166.805450
VEV_5200                 95.548767
VEV_5300                 46.759933
VEV_5400                 15.951917
VEV_5500                  6.641350
VEV_6000                  0.500000
VEV_6500                  0.500000
Name: mid_price, dtype: float64

Total products: 12


In [ ]:
# OU Process Half-Life and Hurst Exponent Analysis

def calculate_halflife_ou(prices):
    """
    Calculate half-life of mean reversion using OLS regression on deviations
    Fits: (x_t - mean) = lambda * (x_{t-1} - mean) + noise
    Half-life = ln(0.5) / ln(lambda)
    """
    prices_clean = prices.replace(0, np.nan).dropna()
    if len(prices_clean) < 20:
        return np.nan
    
    # Calculate deviations from mean
    mean_price = prices_clean.mean()
    deviations = prices_clean - mean_price
    
    # AR(1) regression: dev_t = lambda * dev_{t-1} + error
    X = deviations.iloc[:-1].values.reshape(-1, 1)
    y = deviations.iloc[1:].values
    
    if len(X) < 5:
        return np.nan
    
    # OLS regression - fit line through origin (no intercept for deviations)
    try:
        slope = np.polyfit(X.flatten(), y, 1)[0]
        
        # Mean reversion: 0 < lambda < 1 gives positive half-life
        # Relax constraints to capture more cases
        if abs(slope) < 0.001:
            return np.nan
        
        if slope < 0 or slope > 1.0:
            return np.nan
        
        halflife = np.log(0.5) / np.log(slope)
        
        # Sanity check: half-life should be positive and reasonable
        if halflife < 0.1 or halflife > 10000:
            return np.nan
        
        return halflife
    except Exception as e:
        return np.nan

def calculate_hurst_exponent(series, max_lag=100):
    """
    Calculate Hurst Exponent using Rescaled Range (R/S) Analysis
    H < 0.5: Mean-reverting
    H = 0.5: Random walk
    H > 0.5: Trending
    """
    series = series.dropna()
    if len(series) < 20:
        return np.nan
    
    lags = np.logspace(1, np.log10(min(max_lag, len(series)//2)), 10).astype(int)
    taus = []
    
    for lag in lags:
        if lag >= len(series):
            continue
        
        # Calculate cumulative sum of deviations
        mean = series.mean()
        Y = np.cumsum(series - mean)
        
        # Rescaled range
        R = np.max(Y[:lag]) - np.min(Y[:lag])
        S = series[:lag].std()
        
        if S > 0:
            taus.append(R / S)
    
    if len(taus) < 2:
        return np.nan
    
    # Fit line in log-log space: log(R/S) = H * log(lag)
    coeffs = np.polyfit(np.log(lags[:len(taus)]), np.log(taus), 1)
    hurst = coeffs[0]
    
    return hurst

def find_optimal_rolling_window(series, windows=None):
    """
    Find rolling window that optimizes Hurst exponent (closest to 0.5)
    Returns dataframe with window analysis
    """
    if windows is None:
        windows = np.arange(20, 150, 5)
    
    results = []
    for w in windows:
        if w >= len(series):
            break
        
        rolling_std = series.rolling(window=w).std().dropna()
        hurst = calculate_hurst_exponent(rolling_std)
        
        results.append({
            'window': w,
            'hurst': hurst,
            'distance_from_neutral': abs(hurst - 0.5) if not np.isnan(hurst) else np.nan
        })
    
    return pd.DataFrame(results)

# Analyze each product
print("\n" + "="*80)
print("OU PROCESS ANALYSIS: Half-Life & Optimal Rolling Window")
print("="*80)

analysis_results = []

for product in sorted(df_total['product'].unique()):
    product_data = df_total[df_total['product'] == product].sort_values('timestamp')
    prices = product_data['mid_price'].replace(0, np.nan).dropna()
    
    if len(prices) < 50:
        print(f"\n{product}: Insufficient data ({len(prices)} points)")
        continue
    
    print(f"\n{product}:")
    
    # Half-life calculation with detailed debugging
    halflife = calculate_halflife_ou(prices)
    if not np.isnan(halflife):
        print(f"  Half-Life: {halflife:.2f} periods")
    else:
        # Debug: show why it failed
        mean_price = prices.mean()
        deviations = prices - mean_price
        X = deviations.iloc[:-1].values
        y = deviations.iloc[1:].values
        if len(X) > 0:
            slope = np.polyfit(X, y, 1)[0]
            print(f"  Half-Life: Could not calculate (AR(1) slope: {slope:.4f}, valid range: 0<λ<1)")
        else:
            print(f"  Half-Life: Could not calculate")
    
    # Hurst exponent on price series
    hurst_price = calculate_hurst_exponent(prices)
    if not np.isnan(hurst_price):
        mr_type = "Mean-Reverting (H<0.5)" if hurst_price < 0.5 else "Trending (H>0.5)" if hurst_price > 0.5 else "Random Walk"
        print(f"  Hurst (Price): {hurst_price:.3f} - {mr_type}")
    
    # Optimal window analysis
    window_df = find_optimal_rolling_window(prices, windows=np.arange(20, 150, 5))
    window_df = window_df.dropna(subset=['hurst'])
    
    if not window_df.empty:
        optimal = window_df.loc[window_df['distance_from_neutral'].idxmin()]
        print(f"  Optimal Window: {int(optimal['window'])} periods (Hurst={optimal['hurst']:.3f})")
        
        # Show top 3 windows
        top_3 = window_df.nsmallest(3, 'distance_from_neutral')
        print(f"    Top 3: {', '.join([f'W={int(w)}(H={h:.3f})' for w, h in zip(top_3['window'], top_3['hurst'])])}")
        
        analysis_results.append({
            'Product': product,
            'Half-Life': halflife if not np.isnan(halflife) else 'N/A',
            'Hurst-Price': hurst_price if not np.isnan(hurst_price) else 'N/A',
            'Optimal-Window': int(optimal['window']),
            'Hurst-AtWindow': optimal['hurst']
        })

# Summary table
if analysis_results:
    print("\n" + "="*80)
    print("SUMMARY TABLE")
    print("="*80)
    summary_table = pd.DataFrame(analysis_results)
    print(summary_table.to_string(index=False))


OU PROCESS ANALYSIS: Half-Life & Optimal Rolling Window

HYDROGEL_PACK:
  Half-Life: 0.75 periods
  Hurst (Price): 0.773 - Trending (H>0.5)
  Optimal Window: 75 periods (Hurst=0.506)
    Top 3: W=75(H=0.506), W=50(H=0.482), W=70(H=0.469)

VELVETFRUIT_EXTRACT:
  Half-Life: 0.41 periods
  Hurst (Price): 0.956 - Trending (H>0.5)
  Optimal Window: 145 periods (Hurst=0.330)
    Top 3: W=145(H=0.330), W=140(H=0.060), W=20(H=0.018)

VEV_4000:
  Half-Life: 0.41 periods
  Hurst (Price): 0.952 - Trending (H>0.5)
  Optimal Window: 145 periods (Hurst=0.433)
    Top 3: W=145(H=0.433), W=140(H=0.155), W=135(H=0.012)

VEV_4500:
  Half-Life: 0.41 periods
  Hurst (Price): 0.953 - Trending (H>0.5)
  Optimal Window: 145 periods (Hurst=0.475)
    Top 3: W=145(H=0.475), W=140(H=0.182), W=135(H=0.033)

VEV_5000:
  Half-Life: 0.42 periods
  Hurst (Price): 1.003 - Trending (H>0.5)
  Optimal Window: 20 periods (Hurst=0.064)
    Top 3: W=20(H=0.064), W=25(H=-0.006), W=30(H=-0.090)

VEV_5100:
  Half-Life: 0.44 

/home/dansp/projects/imc_prosperity_tutorial/.venv/lib/python3.14/site-packages/numpy/lib/_polynomial_impl.py:674: RuntimeWarning: invalid value encountered in divide
  lhs /= scale


LinAlgError: SVD did not converge in Linear Least Squares